In [1]:
id_misji = [8861, 8864, 8933, 8934, 8937, 8943, 8964] # + 2143 + 8852 + 8939
dodatek = None
fabula = None
kraina = None

In [2]:
import sys
from pathlib import Path

sciezki_kandydaci = [
    Path.cwd(),
    Path.cwd() / "python-etl",
    Path(r"C:\____Moje-MOJE\MyProjects_4Fun\projects\World of Warcraft\python-etl"),
]

for sciezka in sciezki_kandydaci:
    if (sciezka / "moduly" / "ai_lore_kontekst.py").exists() and str(sciezka) not in sys.path:
        sys.path.insert(0, str(sciezka))

print("sys.path dla python-etl:")
for sciezka in sys.path:
    if "python-etl" in sciezka:
        print("-", sciezka)

sys.path dla python-etl:
- C:\____Moje-MOJE\MyProjects_4Fun\projects\World of Warcraft\python-etl


In [3]:
import json
import os
from sqlalchemy import text

import tiktoken

from langchain_openai import ChatOpenAI

from moduly.ai import validate_quest_content_response
from moduly.ai_lore_kontekst import czytaj_kontekst_lore
# from moduly.ai_modele import llm_translator
from moduly.ai_modele import OPENROUTER_BASE_URL, TEMPERATURE_EDITOR
from moduly.ai_prompty_misje import prompt_editor, tekst_lub_placeholder, editor
from moduly.ai_przyklady_ras_tony_teksty import RACE_STYLES
from moduly.db_core import utworz_engine_do_db
from moduly.utils import (
    formatuj_obsada,
    formatuj_podsumowania_poprzednich_misji,
    formatuj_referencje_de,
    formatuj_slowa_kluczowe,
    formatuj_style_ras,
    hash_do_wsad_json,
)

In [4]:
REASONING_REDAKTORA = {
    "effort": "medium",
    "exclude": True,
}

PARAMETRY_MODELI_REDAKTORA = {
    "qwen/qwen3.7-max": {
        "temperature": 0.60,
        "top_p": 0.95,
        "top_k": 20,
        "reasoning": REASONING_REDAKTORA,
    },
    "moonshotai/kimi-k2.6": {
        "temperature": 0.60,
        "reasoning": REASONING_REDAKTORA,
    },
    "deepseek/deepseek-v4-pro": {
        "temperature": 1.25,
        "reasoning": REASONING_REDAKTORA,
    },
    "google/gemini-3.1-pro-preview": {
        "temperature": 0.60,
        "reasoning": REASONING_REDAKTORA,
    },
    "google/gemini-3.5-flash": {
        "temperature": 0.60,
        "reasoning": REASONING_REDAKTORA,
    },
    "minimax/minimax-m3": {
        "temperature": 0.50,
        "reasoning": REASONING_REDAKTORA,
    },
    "anthropic/claude-sonnet-4.6": {
        "temperature": 0.60,
        "reasoning": REASONING_REDAKTORA,
    },
    "openai/gpt-5.4-mini": {
        "temperature": 0.30,
        "reasoning": REASONING_REDAKTORA,
    },
}


def parametry_dla_modelu(model: str) -> dict:
    return PARAMETRY_MODELI_REDAKTORA.get(
        model,
        {
            "temperature": TEMPERATURE_EDITOR,
            "reasoning": REASONING_REDAKTORA,
        },
    )


def llm_redaktor(model):
    parametry = parametry_dla_modelu(model)
    extra_body = {
        nazwa: wartosc
        for nazwa, wartosc in {
            "top_p": parametry.get("top_p"),
            "top_k": parametry.get("top_k"),
            "reasoning": parametry.get("reasoning"),
        }.items()
        if wartosc is not None
    }

    konfiguracja = {
        "model": model,
        "api_key": os.environ.get("OPENROUTER_API_KEY"),
        "base_url": OPENROUTER_BASE_URL,
        "temperature": parametry.get("temperature", TEMPERATURE_EDITOR),
        "max_retries": 2,
        "default_headers": {
            "X-Title": "World of Warcraft PL Redaction",
        },
    }
    if extra_body:
        konfiguracja["extra_body"] = extra_body

    return ChatOpenAI(**konfiguracja)


llm_translator = llm_redaktor


In [5]:
silnik = utworz_engine_do_db()

In [6]:
misje_do_testu = (
    list(id_misji)
    if isinstance(id_misji, (list, tuple, set))
    else ([id_misji] if id_misji is not None else None)
)
misja_id_podglad = misje_do_testu[0] if misje_do_testu else None

warunki = [
    "m.MISJA_ID_Z_GRY IS NOT NULL",
    "m.MISJA_ID_Z_GRY <> 123456789",
    "z.HTML_SKOMPRESOWANY IS NOT NULL",
]
parametry = {}

if misja_id_podglad is not None:
    warunki.append("m.MISJA_ID_MOJE_PK = :id_misji")
    parametry["id_misji"] = misja_id_podglad
if dodatek is not None:
    warunki.append("m.DODATEK_EN = :dodatek")
    parametry["dodatek"] = dodatek
if fabula is not None:
    warunki.append("m.NAZWA_LINII_FABULARNEJ_EN = :fabula")
    parametry["fabula"] = fabula
if kraina is not None:
    warunki.append("m.KRAINA_EN = :kraina")
    parametry["kraina"] = kraina

q_misja = text(f"""
    WITH najnowsze_zrodlo AS (
        SELECT
            m.MISJA_ID_MOJE_PK,
            z.HTML_SKOMPRESOWANY,
            ROW_NUMBER() OVER (PARTITION BY z.MISJA_ID_MOJE_FK ORDER BY z.DATA_WYSCRAPOWANIA DESC) AS rn
        FROM dbo.ZRODLO_MISJE AS z
        INNER JOIN dbo.MISJE AS m
          ON z.MISJA_ID_MOJE_FK = m.MISJA_ID_MOJE_PK
        WHERE {' AND '.join(warunki)}
    )
    SELECT TOP 1 MISJA_ID_MOJE_PK, HTML_SKOMPRESOWANY
    FROM najnowsze_zrodlo
    WHERE rn = 1
    ORDER BY MISJA_ID_MOJE_PK
""")

with silnik.connect() as conn:
    wiersz = conn.execute(q_misja, parametry).mappings().first()

if wiersz is None:
    raise ValueError("Nie znalazłem misji dla podanych filtrów.")

misja_id = wiersz["MISJA_ID_MOJE_PK"]
zakodowane_dane = wiersz["HTML_SKOMPRESOWANY"]

print(f"Misje do batcha: {misje_do_testu if misje_do_testu is not None else 'wg filtrów'}")
print(f"Wybrana misja do podglądu: {misja_id}")


Misje do batcha: [8861, 8864, 8933, 8934, 8937, 8943, 8964]
Wybrana misja do podglądu: 8861


In [7]:
q_select_npc = text("""
WITH wszystkie_idki AS (
    SELECT tabela_wartosci.ID_NPC
    FROM dbo.MISJE AS m
    CROSS APPLY (VALUES (m.NPC_START_ID), (m.NPC_KONIEC_ID)) AS tabela_wartosci (ID_NPC)
    WHERE m.MISJA_ID_MOJE_PK = :misja_id

    UNION

    SELECT ds.NPC_ID_FK
    FROM dbo.DIALOGI_STATUSY AS ds
    WHERE ds.MISJA_ID_MOJE_FK = :misja_id
),
oczyszczone_dane AS (
    SELECT wi.ID_NPC, ns.STATUS,
    CASE WHEN CHARINDEX('[', ns.NAZWA) > 0 THEN RTRIM(LEFT(ns.NAZWA, CHARINDEX('[', ns.NAZWA) - 1)) ELSE ns.NAZWA END AS CZYSTA_NAZWA
    FROM wszystkie_idki AS wi
    INNER JOIN dbo.NPC_STATUSY AS ns ON wi.ID_NPC = ns.NPC_ID_FK
)
SELECT DISTINCT
    pvt.[0_ORYGINAŁ],
    pvt.[3_ZATWIERDZONO],
    n.PLEC,
    n.RASA
FROM oczyszczone_dane
PIVOT (MAX(CZYSTA_NAZWA) FOR STATUS IN ([0_ORYGINAŁ], [3_ZATWIERDZONO])) AS pvt
LEFT JOIN dbo.NPC AS n
  ON n.NPC_ID_MOJE_PK = pvt.ID_NPC;
""")

q_select_sk = text("""
SELECT sk.SLOWO_EN, sk.SLOWO_PL
FROM dbo.MISJE_SLOWA_KLUCZOWE AS msk
INNER JOIN dbo.SLOWA_KLUCZOWE AS sk
   ON msk.SLOWO_ID = sk.SLOWO_ID_PK
WHERE msk.MISJA_ID_MOJE_FK = :misja_id
""")

q_select_rasa = text("""
WITH teksty_npc AS (
    SELECT
        m.NPC_START_ID AS NPC_ID,
        SUM(LEN(ISNULL(ms.TRESC, N''))) AS ILE_ZNAKOW
    FROM dbo.MISJE_STATUSY AS ms
    INNER JOIN dbo.MISJE AS m
        ON ms.MISJA_ID_MOJE_FK = m.MISJA_ID_MOJE_PK
    WHERE ms.MISJA_ID_MOJE_FK = :misja_id
    AND ms.STATUS = N'0_ORYGINAŁ'
    AND ms.SEGMENT <> N'ZAKOŃCZENIE'
    AND m.NPC_START_ID IS NOT NULL
    GROUP BY m.NPC_START_ID

    UNION ALL

    SELECT
        m.NPC_KONIEC_ID AS NPC_ID,
        SUM(LEN(ISNULL(ms.TRESC, N''))) AS ILE_ZNAKOW
    FROM dbo.MISJE_STATUSY AS ms
    INNER JOIN dbo.MISJE AS m
        ON ms.MISJA_ID_MOJE_FK = m.MISJA_ID_MOJE_PK
    WHERE ms.MISJA_ID_MOJE_FK = :misja_id
    AND ms.STATUS = N'0_ORYGINAŁ'
    AND ms.SEGMENT = N'ZAKOŃCZENIE'
    AND m.NPC_KONIEC_ID IS NOT NULL
    GROUP BY m.NPC_KONIEC_ID

    UNION ALL

    SELECT
        ds.NPC_ID_FK AS NPC_ID,
        SUM(LEN(ISNULL(ds.TRESC, N''))) AS ILE_ZNAKOW
    FROM dbo.DIALOGI_STATUSY AS ds
    WHERE ds.MISJA_ID_MOJE_FK = :misja_id
    AND ds.STATUS = N'0_ORYGINAŁ'
    GROUP BY ds.NPC_ID_FK
),
npc_zsumowane AS (
    SELECT NPC_ID, SUM(ILE_ZNAKOW) AS ILE_ZNAKOW
    FROM teksty_npc
    GROUP BY NPC_ID
)
SELECT n.RASA, SUM(nz.ILE_ZNAKOW) AS ILE_ZNAKOW
FROM npc_zsumowane AS nz
INNER JOIN dbo.NPC AS n
    ON n.NPC_ID_MOJE_PK = nz.NPC_ID
WHERE n.RASA IS NOT NULL
AND n.RASA NOT IN (N'Unknown', '???', '...', 'Brak Danych')
GROUP BY n.RASA
HAVING SUM(nz.ILE_ZNAKOW) > 30
ORDER BY ILE_ZNAKOW DESC;
""")

q_select_fabula = text("""
SELECT NAZWA_LINII_FABULARNEJ_EN
FROM dbo.MISJE
WHERE MISJA_ID_MOJE_PK = :misja_id
""")

q_select_kolejnosc_misja = text("""
SELECT KOLEJNOSC_LINII_FABULARNEJ
FROM dbo.MISJE
WHERE KOLEJNOSC_LINII_FABULARNEJ IS NOT NULL
  AND NAZWA_LINII_FABULARNEJ_EN = :fabula_en
  AND MISJA_ID_MOJE_PK = :misja_id
ORDER BY KOLEJNOSC_LINII_FABULARNEJ ASC
""")

q_select_podsumowanie_poprz_misje = text("""
SELECT
    M.KOLEJNOSC_LINII_FABULARNEJ AS NUMER_MISJI_W_CHAINIE,
    MP.PODSUMOWANIE
FROM dbo.MISJE AS M
INNER JOIN dbo.MISJE_PODSUMOWANIA AS MP
ON M.MISJA_ID_MOJE_PK = MP.MISJA_ID_MOJE_FK
WHERE M.KOLEJNOSC_LINII_FABULARNEJ IS NOT NULL
AND M.NAZWA_LINII_FABULARNEJ_EN = :fabula_en
AND M.KOLEJNOSC_LINII_FABULARNEJ < :kolejnosc_misji
ORDER BY NUMER_MISJI_W_CHAINIE ASC
""")

q_select_referencja_de = text("""
SELECT SEGMENT, NR, TRESC
FROM MISJE_STATUSY
WHERE MISJA_ID_MOJE_FK = :misja_id
  AND STATUS = '4_REFERENCJA'
  AND SEGMENT IN ('TREŚĆ', 'POSTĘP', 'ZAKOŃCZENIE')
ORDER BY
  CASE SEGMENT
    WHEN 'TREŚĆ' THEN 1
    WHEN 'POSTĘP' THEN 2
    WHEN 'ZAKOŃCZENIE' THEN 3
    ELSE 99
  END,
  NR
""")


q_select_obsada = text("""
WITH role_npc AS (
    SELECT m.NPC_START_ID AS NPC_ID, N'START' AS ROLA, 1 AS KOLEJNOSC
    FROM dbo.MISJE AS m
    WHERE m.MISJA_ID_MOJE_PK = :misja_id AND m.NPC_START_ID IS NOT NULL

    UNION ALL

    SELECT m.NPC_KONIEC_ID, N'KONIEC', 2
    FROM dbo.MISJE AS m
    WHERE m.MISJA_ID_MOJE_PK = :misja_id AND m.NPC_KONIEC_ID IS NOT NULL

    UNION ALL

    SELECT DISTINCT ds.NPC_ID_FK, N'DIALOG', 3
    FROM dbo.DIALOGI_STATUSY AS ds
    WHERE ds.MISJA_ID_MOJE_FK = :misja_id AND ds.NPC_ID_FK IS NOT NULL
),
nazwy_npc AS (
    SELECT
        ns.NPC_ID_FK,
        MAX(CASE WHEN CHARINDEX('[', ns.NAZWA) > 0
                 THEN RTRIM(LEFT(ns.NAZWA, CHARINDEX('[', ns.NAZWA) - 1))
                 ELSE ns.NAZWA END) AS CZYSTA_NAZWA
    FROM dbo.NPC_STATUSY AS ns
    WHERE ns.STATUS = N'0_ORYGINAŁ'
    GROUP BY ns.NPC_ID_FK
)
SELECT
    rn.ROLA,
    rn.KOLEJNOSC,
    nz.CZYSTA_NAZWA AS NAZWA_EN,
    n.RASA
FROM role_npc AS rn
LEFT JOIN nazwy_npc AS nz ON nz.NPC_ID_FK = rn.NPC_ID
LEFT JOIN dbo.NPC AS n ON n.NPC_ID_MOJE_PK = rn.NPC_ID
ORDER BY rn.KOLEJNOSC, nz.CZYSTA_NAZWA;
""")


In [8]:
with silnik.connect() as conn:
    misja_referencja_de_wiersze = conn.execute(q_select_referencja_de, {"misja_id": misja_id}).fetchall()
    fabula_z_bazy = conn.execute(q_select_fabula, {"misja_id": misja_id}).first()
    fabula_en = fabula_z_bazy[0] if fabula_z_bazy else None
    npc_z_bazy = conn.execute(q_select_npc, {"misja_id": misja_id}).all()
    obsada_z_bazy = conn.execute(q_select_obsada, {"misja_id": misja_id}).all()
    slowa_kluczowe_z_bazy = conn.execute(q_select_sk, {"misja_id": misja_id}).all()
    wybrane_rasy_z_bazy = conn.execute(q_select_rasa, {"misja_id": misja_id}).all()
    kolejnosc_misja = conn.execute(q_select_kolejnosc_misja, {
        "misja_id": misja_id,
        "fabula_en": fabula_en
    }).first()
    kolejnosc_misji = kolejnosc_misja[0] if kolejnosc_misja else None

    podsumowania_poprzednich_misji = []
    if kolejnosc_misji is not None:
        podsumowania_poprzednich_misji = conn.execute(q_select_podsumowanie_poprz_misje, {
            "fabula_en": fabula_en,
            "kolejnosc_misji": kolejnosc_misji
        }).all()

    context_lore_text = czytaj_kontekst_lore(conn, misja_id)

misja_referencja_de = formatuj_referencje_de(misja_referencja_de_wiersze)

wsad_npc = set(n for n in npc_z_bazy)
wsad_sk = set(s for s in slowa_kluczowe_z_bazy)
wsad_json = hash_do_wsad_json(zakodowane_dane, jezyk="EN")
wsad_wybrane_rasy_opis = set(r[0] for r in wybrane_rasy_z_bazy)

wsad_podsumowania_poprzednich_misji_w_chainie = formatuj_podsumowania_poprzednich_misji(
    podsumowania_poprzednich_misji,
    kolejnosc_misji,
)

txt_npc = "\n".join(
    [(
        f"- nazwa_en: {n[0]}\n"
        f"  nazwa_pl: {n[1]}\n"
        f"  plec: {n[2] or 'Unknown'}\n"
        f"  rasa: {n[3] or 'Unknown'}"
    )
        for n in sorted(wsad_npc, key=lambda x: (x[0] or "", x[1] or ""))
        if n[0] and n[1]
    ]
)

txt_sk = formatuj_slowa_kluczowe(wsad_sk)

txt_rasy_tlumacz = formatuj_style_ras(RACE_STYLES, wsad_wybrane_rasy_opis, etap="tlumacz")
txt_rasy_redaktor = formatuj_style_ras(RACE_STYLES, wsad_wybrane_rasy_opis, etap="redaktor")
txt_obsada = formatuj_obsada(obsada_z_bazy)

print("Dane gotowe.")
print(f"typ misja_referencja_de: {type(misja_referencja_de).__name__}")
print(f"NPC: {len(wsad_npc)}, słowa kluczowe: {len(wsad_sk)}, rasy: {len(wsad_wybrane_rasy_opis)}")

Dane gotowe.
typ misja_referencja_de: str
NPC: 5, słowa kluczowe: 14, rasy: 3


In [9]:
if not isinstance(misja_referencja_de, str):
    misja_referencja_de = formatuj_referencje_de(misja_referencja_de)

folder_tlumacza = Path(
    r"C:\____Moje-MOJE\MyProjects_4Fun\projects\World of Warcraft\_testy_tlumacz\Zrzuty\03_Kolejny test tłumacza"
) / str(misja_id)
plik_tlumacza = folder_tlumacza / "02_kandydat_3.txt"

if not plik_tlumacza.exists():
    raise FileNotFoundError(f"Nie znaleziono pliku z draftem tłumacza: {plik_tlumacza}")

def odetnij_naglowek_kandydata(tresc: str) -> str:
    tresc = tresc.strip()
    if tresc.startswith(("KANDYDAT:", "MODEL:")) and "\n\n" in tresc:
        return tresc.split("\n\n", 1)[1].strip()
    return tresc

tekst_przetlumaczony = odetnij_naglowek_kandydata(
    plik_tlumacza.read_text(encoding="utf-8")
)

print(f"typ misja_referencja_de przed promptem: {type(misja_referencja_de).__name__}")
print(f"draft tłumacza: {plik_tlumacza}")

wiadomosci_redaktora = prompt_editor.format_messages(
    tekst_oryginalny=tekst_lub_placeholder(wsad_json, "{}"),
    tekst_przetlumaczony=tekst_lub_placeholder(tekst_przetlumaczony, "{}"),
    tekst_pomocniczy=tekst_lub_placeholder(misja_referencja_de, "- brak wersji niemieckiej dla tej misji"),
    kontekst_rag=tekst_lub_placeholder(context_lore_text, "- brak kontekstu dla tej misji"),
    wytyczne_rasy=tekst_lub_placeholder(txt_rasy_redaktor, "- brak wytycznych dla tej/tych ras"),
    tekst_npc=tekst_lub_placeholder(txt_npc, "- brak mapowań NPC dla tej misji"),
    tekst_slowa_kluczowe=tekst_lub_placeholder(txt_sk, "- brak mapowań słów kluczowych dla tej misji"),
    podsumowania_poprzednich_misji_w_chainie=tekst_lub_placeholder(
        wsad_podsumowania_poprzednich_misji_w_chainie,
        "- jest to zwykła misja nie będąca w żadnym chainie albo pierwsza misja w chainie"
    )
)

typ misja_referencja_de przed promptem: str
draft tłumacza: C:\____Moje-MOJE\MyProjects_4Fun\projects\World of Warcraft\_testy_tlumacz\Zrzuty\03_Kolejny test tłumacza\8861\02_kandydat_3.txt


In [10]:
# print(tekst_przetlumaczony)
# print(wiadomosci_redaktora)

In [11]:
import concurrent.futures

MODELE_REDAKTORA = {
    "1": "qwen/qwen3.7-max",
    # "2": "moonshotai/kimi-k2.6",
    # "3": "deepseek/deepseek-v4-pro",
    # "4": "google/gemini-3.1-pro-preview",
    # "5": "google/gemini-3.5-flash",
    # "6": "minimax/minimax-m3",
    # "7": "anthropic/claude-sonnet-4.6",
    "8": "openai/gpt-5.4-mini"
}

folder_testow_baza = Path(
    r"C:\____Moje-MOJE\MyProjects_4Fun\projects\World of Warcraft\_testy_redaktor"
)
folder_tlumacza_baza = Path(
    r"C:\____Moje-MOJE\MyProjects_4Fun\projects\World of Warcraft\_testy_tlumacz\Zrzuty\03_Kolejny test tłumacza"
)

misje_batch = misje_do_testu if misje_do_testu is not None else [misja_id]

q_zrodlo_misji_batch = text("""
    WITH najnowsze_zrodlo AS (
        SELECT
            m.MISJA_ID_MOJE_PK,
            z.HTML_SKOMPRESOWANY,
            ROW_NUMBER() OVER (PARTITION BY z.MISJA_ID_MOJE_FK ORDER BY z.DATA_WYSCRAPOWANIA DESC) AS rn
        FROM dbo.ZRODLO_MISJE AS z
        INNER JOIN dbo.MISJE AS m
          ON z.MISJA_ID_MOJE_FK = m.MISJA_ID_MOJE_PK
        WHERE m.MISJA_ID_MOJE_PK = :misja_id
          AND m.MISJA_ID_Z_GRY IS NOT NULL
          AND m.MISJA_ID_Z_GRY <> 123456789
          AND z.HTML_SKOMPRESOWANY IS NOT NULL
    )
    SELECT TOP 1 MISJA_ID_MOJE_PK, HTML_SKOMPRESOWANY
    FROM najnowsze_zrodlo
    WHERE rn = 1
""")

def odetnij_naglowek_kandydata(tresc: str) -> str:
    tresc = tresc.strip()
    if tresc.startswith(("KANDYDAT:", "MODEL:")) and "\n\n" in tresc:
        return tresc.split("\n\n", 1)[1].strip()
    return tresc

def _plikowy_model(model):
    return model.replace("/", "∕")

def _przygotuj_dane_misji(misja_id_batch):
    with silnik.connect() as conn:
        wiersz = conn.execute(q_zrodlo_misji_batch, {"misja_id": misja_id_batch}).mappings().first()
        if wiersz is None:
            raise ValueError(f"Nie znalazłem źródła misji: {misja_id_batch}")

        misja_referencja_de_wiersze = conn.execute(q_select_referencja_de, {"misja_id": misja_id_batch}).fetchall()
        fabula_z_bazy = conn.execute(q_select_fabula, {"misja_id": misja_id_batch}).first()
        fabula_en = fabula_z_bazy[0] if fabula_z_bazy else None
        npc_z_bazy = conn.execute(q_select_npc, {"misja_id": misja_id_batch}).all()
        slowa_kluczowe_z_bazy = conn.execute(q_select_sk, {"misja_id": misja_id_batch}).all()
        wybrane_rasy_z_bazy = conn.execute(q_select_rasa, {"misja_id": misja_id_batch}).all()
        kolejnosc_misja = conn.execute(q_select_kolejnosc_misja, {
            "misja_id": misja_id_batch,
            "fabula_en": fabula_en,
        }).first()
        kolejnosc_misji = kolejnosc_misja[0] if kolejnosc_misja else None

        podsumowania_poprzednich_misji = []
        if kolejnosc_misji is not None:
            podsumowania_poprzednich_misji = conn.execute(q_select_podsumowanie_poprz_misje, {
                "fabula_en": fabula_en,
                "kolejnosc_misji": kolejnosc_misji,
            }).all()

        context_lore_text = czytaj_kontekst_lore(conn, misja_id_batch)

    misja_referencja_de = formatuj_referencje_de(misja_referencja_de_wiersze)
    wsad_npc = set(n for n in npc_z_bazy)
    wsad_sk = set(s for s in slowa_kluczowe_z_bazy)
    wsad_json = hash_do_wsad_json(wiersz["HTML_SKOMPRESOWANY"], jezyk="EN")
    wsad_wybrane_rasy_opis = set(r[0] for r in wybrane_rasy_z_bazy)
    wsad_podsumowania_poprzednich_misji_w_chainie = formatuj_podsumowania_poprzednich_misji(
        podsumowania_poprzednich_misji,
        kolejnosc_misji,
    )

    txt_npc = "\n".join(
        [(
            f"- nazwa_en: {n[0]}\n"
            f"  nazwa_pl: {n[1]}\n"
            f"  plec: {n[2] or 'Unknown'}\n"
            f"  rasa: {n[3] or 'Unknown'}"
        )
            for n in sorted(wsad_npc, key=lambda x: (x[0] or "", x[1] or ""))
            if n[0] and n[1]
        ]
    )
    txt_sk = formatuj_slowa_kluczowe(wsad_sk)
    txt_rasy_redaktor = formatuj_style_ras(RACE_STYLES, wsad_wybrane_rasy_opis, etap="redaktor")

    plik_tlumacza = folder_tlumacza_baza / str(misja_id_batch) / "02_kandydat_3.txt"
    if not plik_tlumacza.exists():
        raise FileNotFoundError(f"Nie znaleziono pliku z draftem tłumacza: {plik_tlumacza}")

    tekst_przetlumaczony = odetnij_naglowek_kandydata(
        plik_tlumacza.read_text(encoding="utf-8")
    )

    wiadomosci_redaktora = prompt_editor.format_messages(
        tekst_oryginalny=tekst_lub_placeholder(wsad_json, "{}"),
        tekst_przetlumaczony=tekst_lub_placeholder(tekst_przetlumaczony, "{}"),
        tekst_pomocniczy=tekst_lub_placeholder(misja_referencja_de, "- brak wersji niemieckiej dla tej misji"),
        kontekst_rag=tekst_lub_placeholder(context_lore_text, "- brak kontekstu dla tej misji"),
        wytyczne_rasy=tekst_lub_placeholder(txt_rasy_redaktor, "- brak wytycznych dla tej/tych ras"),
        tekst_npc=tekst_lub_placeholder(txt_npc, "- brak mapowań NPC dla tej misji"),
        tekst_slowa_kluczowe=tekst_lub_placeholder(txt_sk, "- brak mapowań słów kluczowych dla tej misji"),
        podsumowania_poprzednich_misji_w_chainie=tekst_lub_placeholder(
            wsad_podsumowania_poprzednich_misji_w_chainie,
            "- jest to zwykła misja nie będąca w żadnym chainie albo pierwsza misja w chainie",
        ),
    )

    folder_testow = folder_testow_baza / str(misja_id_batch)
    folder_testow.mkdir(parents=True, exist_ok=True)

    prompt_redaktora_pelny = "\n\n".join(
        f"--- {msg.type.upper()} ---\n{msg.content}"
        for msg in wiadomosci_redaktora
    )
    (folder_testow / "00_prompt_redaktora.txt").write_text(
        prompt_redaktora_pelny,
        encoding="utf-8",
    )
    (folder_testow / "01_draft_tlumacza.txt").write_text(
        tekst_przetlumaczony,
        encoding="utf-8",
    )

    return {
        "misja_id": misja_id_batch,
        "folder_testow": folder_testow,
        "wsad_json": wsad_json,
        "tekst_przetlumaczony": tekst_przetlumaczony,
        "misja_referencja_de": misja_referencja_de,
        "context_lore_text": context_lore_text,
        "txt_rasy_redaktor": txt_rasy_redaktor,
        "txt_npc": txt_npc,
        "txt_sk": txt_sk,
        "wsad_podsumowania_poprzednich_misji_w_chainie": wsad_podsumowania_poprzednich_misji_w_chainie,
    }

def _zapisz_wynik(dane, nr_modelu, model, tresc):
    folder_testow = dane["folder_testow"]
    model_w_nazwie_pliku = _plikowy_model(model)

    (folder_testow / f"02_{model_w_nazwie_pliku}.txt").write_text(
        f"MODEL: {model}\n\n{tresc}",
        encoding="utf-8",
    )
    (folder_testow / f"03_redaktor_{nr_modelu}.txt").write_text(
        tresc,
        encoding="utf-8",
    )

def _uruchom_model(dane, nr_modelu, model):
    misja_id_batch = dane["misja_id"]
    try:
        parametry = parametry_dla_modelu(model)
        print(f"Start: misja {misja_id_batch}, redaktor_{nr_modelu} -> {model} | parametry={parametry}")
        _redaktor = llm_redaktor(model)

        result_editor = editor(
            llm=_redaktor,
            tekst_oryginalny=dane["wsad_json"],
            tekst_przetlumaczony=dane["tekst_przetlumaczony"],
            tekst_pomocniczy=dane["misja_referencja_de"],
            kontekst_rag=dane["context_lore_text"],
            wytyczne_rasy=dane["txt_rasy_redaktor"],
            tekst_npc=dane["txt_npc"],
            tekst_slowa_kluczowe=dane["txt_sk"],
            podsumowania_poprzednich_misji_w_chainie=dane["wsad_podsumowania_poprzednich_misji_w_chainie"],
        )

        if result_editor["parsing_error"] is not None:
            raise result_editor["parsing_error"]

        if result_editor["parsed"] is None:
            raise ValueError("Redaktor nie zwrócił poprawnie sparsowanego JSON-a.")

        validate_quest_content_response(
            result_editor["parsed"],
            misja_id=misja_id_batch,
            stage="editor",
        )

        tresc = json.dumps(result_editor["parsed"], indent=2, ensure_ascii=False)
        status = "OK"
    except Exception as e:
        tresc = f"BŁĄD PODCZAS GENEROWANIA - {type(e).__name__}: {e}"
        status = "BŁĄD"

    _zapisz_wynik(dane, nr_modelu, model, tresc)
    return misja_id_batch, nr_modelu, model, status

wyniki_batcha = []

for misja_id_batch in misje_batch:
    print("=" * 120)
    print(f"Misja {misja_id_batch}: przygotowuję dane i prompt redaktora")
    dane_misji = _przygotuj_dane_misji(misja_id_batch)

    with concurrent.futures.ThreadPoolExecutor(max_workers=len(MODELE_REDAKTORA)) as executor:
        futures = [
            executor.submit(_uruchom_model, dane_misji, nr_modelu, model)
            for nr_modelu, model in MODELE_REDAKTORA.items()
        ]

        for future in concurrent.futures.as_completed(futures):
            misja_id_wynik, nr_modelu, model, status = future.result()
            wyniki_batcha.append((misja_id_wynik, nr_modelu, model, status))
            print(f"{status}: misja {misja_id_wynik}, redaktor_{nr_modelu} -> {model}")

print("=" * 120)
print(f"Gotowe. Misje: {len(misje_batch)}, modele na misję: {len(MODELE_REDAKTORA)}, zadania: {len(wyniki_batcha)}")
print(f"Folder bazowy: {folder_testow_baza}")


Misja 8861: przygotowuję dane i prompt redaktora
Start: misja 8861, redaktor_1 -> qwen/qwen3.7-max | parametry={'temperature': 0.6, 'top_p': 0.95, 'top_k': 20, 'reasoning': {'effort': 'medium', 'exclude': True}}
Start: misja 8861, redaktor_8 -> openai/gpt-5.4-mini | parametry={'temperature': 0.3, 'reasoning': {'effort': 'medium', 'exclude': True}}
OK: misja 8861, redaktor_8 -> openai/gpt-5.4-mini
OK: misja 8861, redaktor_1 -> qwen/qwen3.7-max
Misja 8864: przygotowuję dane i prompt redaktora
Start: misja 8864, redaktor_1 -> qwen/qwen3.7-max | parametry={'temperature': 0.6, 'top_p': 0.95, 'top_k': 20, 'reasoning': {'effort': 'medium', 'exclude': True}}
Start: misja 8864, redaktor_8 -> openai/gpt-5.4-mini | parametry={'temperature': 0.3, 'reasoning': {'effort': 'medium', 'exclude': True}}
OK: misja 8864, redaktor_8 -> openai/gpt-5.4-mini
OK: misja 8864, redaktor_1 -> qwen/qwen3.7-max
Misja 8933: przygotowuję dane i prompt redaktora
Start: misja 8933, redaktor_1 -> qwen/qwen3.7-max | param

In [12]:
# print("=" * 120)
# print("PROMPT REDAKTORA, BEZ URUCHAMIANIA REDAKCJI")
# print("=" * 120)

# for msg in wiadomosci_redaktora:
#     print(f"\n--- {msg.type.upper()} ---\n")
#     print(msg.content)
